In [474]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("D:/Python/libraries/smart_home_prodaja_2025.csv")

df

,Datum,Proizvod,Grad,Cena_EUR,Količina,Ukupna_Zarada
0,2025-04-13,Pametna Brava Locky,Beograd,237.07,5.0,1185.35
1,2025-05-02,Senzor Pokreta,Beograd,408.87,4.0,1635.48
2,2025-08-03,Robot Usisivač X1,Beograd,261.38,8.0,2091.04
3,2025-01-05,Robot Usisivač X1,Novi Sad,301.33,8.0,2410.64
4,2025-01-05,Pametna Sijalica RGB,Subotica,249.26,1.0,249.26
...,...,...,...,...,...,...
441,2025-01-04,Prečišćivač AirPro,Kragujevac,190.17,9.0,1711.53
442,2025-01-16,Senzor Pokreta,Niš,195.98,2.0,391.96
443,2025-10-07,Robot Usisivač X1,Subotica,371.59,2.0,743.18
444,2025-12-02,Pametna Sijalica RGB,Novi Sad,10.54,3.0,31.62


## Distribucija elemenata

#### Uporedi na jednom histogramu 'Robot Usisivač X1' (skup proizvod) i 'Pametnu Sijalicu RGB' (jeftin proizvod). Baš me zanima kako izgledaju njihovi "bregovi" zarade na istoj skali.

In [475]:
mask = (df["Proizvod"] == "Pametna Sijalica RGB") | (df["Proizvod"] == "Robot Usisivač X1")
df_filter = df[mask]

fig = px.histogram(df_filter, 
                   x="Ukupna_Zarada", 
                   nbins=20, 
                   color="Proizvod",
                   color_discrete_map={"Robot Usisivač X1": "#42FF2A", "Pametna Sijalica RGB": "#FFF837"},
                   template="plotly_dark")

fig.update_xaxes(title="Ukupna Zarada",
                 title_font=dict(size=17, color="lightblue"),
                 dtick=200,
                 tickangle=30,
                 tickfont=dict(size=12, color='gray'))

fig.update_yaxes(title="Distribucija",
                 title_font=dict(size=17, color="lightblue"),
                 range=[0, 35],
                 tickfont=dict(size=12, color='gray'))

fig.update_traces(hovertemplate= 
                  "<b>Proizvod</b>: %{fullData.name}<br>"
                  "<b>Zarada</b>: %{x}<br>"
                  "Broj: %{y}" 
                  "<extra></extra>",
                  xbins=dict(start=0, end=4000, size = 200))

fig.update_layout(title=dict(text="Distribucija ukupne zarade usisivača i sijalice",
                             x=0.5,
                             xanchor="center" ,
                            font=dict(size=20, color="lightblue")), 
                legend=dict(orientation="h",
                             x=0.5,
                             xanchor="center",
                             y=0.98,
                             yanchor="bottom",),            
                bargap=0.1)

fig.show()

## Geografija prodaje

#### Napravi bar chart koji pokazuje ukupnu zaradu po gradovima. Želim da vidim koji grad je lider u automatizaciji doma.


In [476]:
df_group = df.groupby("Grad").agg(suma = ("Ukupna_Zarada", "sum")).reset_index().sort_values(by="suma", ascending=False)

fig = px.bar(df_group, 
             x = "suma", 
             y = "Grad", 
             color= "suma",
             color_continuous_scale=["#FF7722", "#EDF724"],
             text_auto=",.0f",
             template="plotly_dark")

fig.update_layout(coloraxis_showscale=False,
                  title=dict(text="Ukupna zarada po gradovima", x=0.5, xanchor="center", font=dict(size=22, color="orange")))

fig.update_xaxes(title="Ukupna Zarada (RSD)", 
                 title_font=dict(size=18, color="orange"), 
                 tickformat=",.0f", 
                 dtick=10000,
                 tickangle=30,
                 range=[0, 125000])

fig.update_yaxes(title="Grad", title_font=dict(size=18, color="orange"))

fig.update_traces(hovertemplate="<b>Ukupna Zarada</b>: %{x:,.0f}<br><b>Grad</b>: %{y}")

fig.show()

## Vremenska prognoza

#### Izvuci mesečni trend zarade. Da li ljudi više kupuju pametne uređaje zimi ili leti?

In [477]:
df["Datum"] = pd.to_datetime(df["Datum"], errors="coerce") # Iz nekoga razloga moram ovo

df["Mesec"] = df["Datum"].dt.month

df_group = df.groupby("Mesec").agg(suma = ("Količina", "sum")).reset_index()

fig = px.line(df_group, 
              x = "Mesec", 
              y = "suma",
              markers=True,
              template="plotly_dark")

fig.update_layout(title=dict(text="Vremenska prognoza - Kada se najviše kupuje?", x=0.5, xanchor="center", font=dict(size=22, color="#00FFF7")))

fig.update_xaxes(title="Mesec", 
                 title_font=dict(size=18, color="#00FFF7"), 
                 dtick=1,
                 range=[1, 12])

fig.update_yaxes(title="Broj prodatih uređaja", title_font=dict(size=18, color="#00FFF7"))

fig.update_traces(line_color="#00D9FF")

fig.show()